# Семинар / ДЗ: деревья, случайный лес и бустинг на MNIST
**Выполнил:** Хритинин Д. А.

**Задание:**
1. Загрузить датасет MNIST.
2. Проанализировать сбалансированность, вывести примеры.
3. Обучить дерево без ограничения глубины, дерево с ограничением глубины, случайный лес и бустинг.
4. Сравнить результаты.

## 1. Загрузка данных

In [ ]:
%pip install kagglehub --quiet
import kagglehub

path = kagglehub.dataset_download("hojjatk/mnist-dataset")
print("Path to dataset files:", path)

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Файлы в формате idx: заголовок 16 байт у картинок и 8 байт у меток
def read_images(f):
    with open(f, "rb") as fh:
        data = np.frombuffer(fh.read(), np.uint8, offset=16)
    return data.reshape(-1, 784)

def read_labels(f):
    with open(f, "rb") as fh:
        return np.frombuffer(fh.read(), np.uint8, offset=8)

def find(pattern):
    return sorted(glob.glob(os.path.join(path, "**", pattern), recursive=True))[0]

X_train = read_images(find("train-images*idx3*")) / 255.0
y_train = read_labels(find("train-labels*idx1*"))
X_test  = read_images(find("t10k-images*idx3*")) / 255.0
y_test  = read_labels(find("t10k-labels*idx1*"))

print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. Сбалансированность классов и примеры

In [ ]:
counts = pd.Series(y_train).value_counts().sort_index()
print(pd.DataFrame({"кол-во": counts, "доля, %": (counts / len(y_train) * 100).round(2)}))

counts.plot(kind="bar", figsize=(8, 4), title="Количество примеров каждой цифры (train)")
plt.xlabel("Цифра"); plt.ylabel("Количество"); plt.xticks(rotation=0); plt.show()

print("Отношение самого большого класса к самому маленькому:", round(counts.max() / counts.min(), 2))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for d, ax in enumerate(axes.flat):
    ax.imshow(X_train[y_train == d][0].reshape(28, 28), cmap="gray")
    ax.set_title(f"Цифра {d}"); ax.axis("off")
plt.suptitle("Примеры изображений"); plt.tight_layout(); plt.show()

## 3. Обучение моделей
Для бустинга используется `HistGradientBoostingClassifier`: обычный `GradientBoostingClassifier` на 60 000 × 784 обучается очень долго.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

models = {
    "Дерево (без ограничения)": DecisionTreeClassifier(random_state=42),
    "Дерево (max_depth=10)":    DecisionTreeClassifier(max_depth=10, random_state=42),
    "Случайный лес":            RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "Бустинг (HistGB)":         HistGradientBoostingClassifier(max_iter=100, random_state=42),
}

results = []
for name, model in models.items():
    t = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t
    results.append({
        "Модель": name,
        "Train acc": round(accuracy_score(y_train, model.predict(X_train)), 4),
        "Test acc":  round(accuracy_score(y_test, model.predict(X_test)), 4),
        "Время обучения, с": round(fit_time, 1),
    })
    print(f"{name}: готово за {fit_time:.1f} с")

## 4. Сравнение

In [ ]:
res = pd.DataFrame(results)
res["Train - Test"] = (res["Train acc"] - res["Test acc"]).round(4)
res

In [ ]:
res.plot(x="Модель", y=["Train acc", "Test acc"], kind="bar", figsize=(9, 5), ylim=(0.8, 1.01), rot=15)
plt.title("Accuracy моделей на MNIST"); plt.grid(axis="y", alpha=0.3); plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
best_name = res.loc[res["Test acc"].idxmax(), "Модель"]
ConfusionMatrixDisplay.from_estimator(models[best_name], X_test, y_test, cmap="Blues")
plt.title(f"Матрица ошибок: {best_name}"); plt.show()

## Выводы
*(цифры ниже приблизительные; сверь с таблицей выше после запуска)*

- **Баланс классов:** на каждую цифру приходится примерно 9–11% выборки (от ~5400 примеров у «5» до ~6700 у «1»). Классы почти сбалансированы, поэтому accuracy здесь адекватная метрика.
- **Дерево без ограничения глубины:** train accuracy = 1.0, test ≈ 0.87–0.88. Большой разрыв означает сильное переобучение: дерево «запомнило» обучающую выборку.
- **Дерево с max_depth=10:** разрыв между train и test меньше, но и test-качество не выше (≈ 0.86): ограниченное дерево слишком простое для 784 пиксельных признаков, т.е. недообучается.
- **Случайный лес:** test ≈ 0.97. Усреднение ответов многих деревьев, обученных на разных подвыборках объектов и признаков, сильно снижает разброс (variance) модели.
- **Бустинг:** test ≈ 0.97–0.98, обычно лучший результат. Каждое следующее дерево исправляет ошибки предыдущих, снижая смещение (bias). Обучается дольше всех.
- **Итог:** ансамбли дают прирост около 10 п.п. по сравнению с одиночным деревом. Одно дерево на изображениях работает плохо: его либо приходится ограничивать (недообучение), либо оно переобучается.